# Round 1 Experiment Tracker

This notebook reads `../results/experiments.csv` and turns the logged experiment rows into a compact comparison dashboard.

Use it to:
- inspect the latest `round1` runs
- compare validation-best and test metrics side by side
- check which settings improved `IoU` and `F1`
- compare `round1` runs against older non-round1 baselines when they exist

Tip: pass a stable `--experiment-name` when you run `train.py` and `test.py`. If that name contains `round1`, this notebook will automatically group it into the current round.

In [ ]:
import csv
import html
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import HTML, display

RESULTS_CSV = Path("..") / "results" / "experiments.csv"
ROUND1_KEYWORD = "round1"
plt.rcParams["figure.dpi"] = 120

## 1. Load Logged Rows

These helpers keep the notebook dependency-light, so it works even without `pandas`.

In [ ]:
NUMERIC_FIELDS = {
    "img_size",
    "batch_size",
    "epochs",
    "lr",
    "num_workers",
    "foreground_crop_prob",
    "foreground_crop_min_scale",
    "foreground_crop_max_scale",
    "pos_weight",
    "tversky_alpha",
    "tversky_beta",
    "tversky_gamma",
    "best_epoch",
    "dataset_size",
    "metric_loss",
    "metric_iou",
    "metric_f1",
}


def parse_optional_number(value):
    if value in ("", None):
        return None
    try:
        return float(value)
    except ValueError:
        return value


def row_label(row):
    if row.get("experiment_name"):
        return row["experiment_name"]
    checkpoint = row.get("checkpoint_path", "")
    return Path(checkpoint).stem if checkpoint else row.get("run_id", "unknown")


def load_experiments(csv_path):
    if not csv_path.exists():
        print(f"No experiment CSV found yet at {csv_path.resolve()}")
        return []

    rows = []
    with csv_path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            parsed = dict(row)
            for key in NUMERIC_FIELDS:
                parsed[key] = parse_optional_number(parsed.get(key))
            parsed["label"] = row_label(parsed)
            parsed["is_round1"] = ROUND1_KEYWORD in (
                (parsed.get("experiment_name") or "") + " " + parsed.get("checkpoint_path", "")
            ).lower()
            rows.append(parsed)
    return rows


def latest_rows(rows, keys=("experiment_name", "stage", "split")):
    latest = {}
    for row in sorted(rows, key=lambda r: r.get("timestamp_utc", "")):
        key = tuple(row.get(k, "") for k in keys)
        latest[key] = row
    return list(latest.values())


def duplicate_groups(rows, keys=("experiment_name", "stage", "split")):
    grouped = {}
    for row in rows:
        key = tuple(row.get(k, "") for k in keys)
        grouped.setdefault(key, []).append(row)

    duplicates = []
    for key, group in grouped.items():
        if len(group) <= 1:
            continue
        ordered = sorted(group, key=lambda r: r.get("timestamp_utc", ""))
        newest = ordered[-1]
        duplicates.append(
            {
                "experiment_name": key[0],
                "stage": key[1],
                "split": key[2],
                "count": len(group),
                "latest_timestamp_utc": newest.get("timestamp_utc", ""),
            }
        )

    return sorted(
        duplicates,
        key=lambda r: (r.get("experiment_name", ""), r.get("stage", ""), r.get("split", "")),
    )


def fmt(value):
    if value is None:
        return ""
    if isinstance(value, float):
        return f"{value:.4f}"
    return str(value)


def display_rows(rows, columns):
    if not rows:
        print("No rows to display.")
        return

    header = "".join(f"<th>{html.escape(col)}</th>" for col in columns)
    body = []
    for row in rows:
        cells = "".join(
            f"<td>{html.escape(fmt(row.get(col, '')))}</td>"
            for col in columns
        )
        body.append(f"<tr>{cells}</tr>")

    table_html = (
        "<table>"
        "<thead><tr>" + header + "</tr></thead>"
        "<tbody>" + "".join(body) + "</tbody>"
        "</table>"
    )
    display(HTML(table_html))


def sorted_stage_rows(rows, stage):
    return sorted(
        [row for row in rows if row.get("stage") == stage],
        key=lambda r: (r.get("metric_iou") is None, -(r.get("metric_iou") or -1e9)),
    )


def plot_stage_metric(ax, rows, stage, metric_key, title):
    stage_rows = sorted_stage_rows(rows, stage)
    if not stage_rows:
        ax.set_title(title)
        ax.text(0.5, 0.5, "No rows", ha="center", va="center")
        ax.axis("off")
        return

    labels = [row["label"] for row in stage_rows]
    values = [row.get(metric_key) or 0.0 for row in stage_rows]
    positions = list(range(len(stage_rows)))

    ax.barh(positions, values, color="#d95f02" if stage == "test" else "#1b9e77")
    ax.set_yticks(positions)
    ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(metric_key.replace("metric_", ""))

    for pos, value in zip(positions, values):
        ax.text(value + 0.003, pos, f"{value:.3f}", va="center", fontsize=8)


def best_baseline(rows, stage, split, model_name="Unet"):
    candidates = [
        row for row in rows
        if row.get("stage") == stage
        and row.get("split") == split
        and row.get("model_name") == model_name
        and not row.get("is_round1")
        and row.get("metric_iou") is not None
    ]
    if not candidates:
        return None
    return max(candidates, key=lambda row: row["metric_iou"])

In [ ]:
all_rows = load_experiments(RESULTS_CSV)
latest_all_rows = latest_rows(all_rows)
duplicate_row_groups = duplicate_groups(all_rows)

round1_rows = [row for row in latest_all_rows if row.get("is_round1")]
active_rows = round1_rows if round1_rows else latest_all_rows

print(f"CSV path: {RESULTS_CSV.resolve()}")
print(f"Total logged rows: {len(all_rows)}")
print(f"Latest deduplicated rows: {len(latest_all_rows)}")
if duplicate_row_groups:
    print(f"Duplicate experiment keys detected: {len(duplicate_row_groups)}")
else:
    print("No duplicate experiment keys detected.")
if round1_rows:
    print(f"Using {len(round1_rows)} round1-tagged rows.")
else:
    print("No round1-tagged rows found yet. Falling back to all latest rows.")

if duplicate_row_groups:
    display_rows(
        duplicate_row_groups,
        columns=[
            "experiment_name",
            "stage",
            "split",
            "count",
            "latest_timestamp_utc",
        ],
    )

display_rows(
    sorted(active_rows, key=lambda row: (row.get("stage", ""), row.get("timestamp_utc", ""))),
    columns=[
        "timestamp_utc",
        "label",
        "stage",
        "split",
        "model_name",
        "img_size",
        "augmentation_profile",
        "foreground_crop_prob",
        "loss_name",
        "metric_iou",
        "metric_f1",
    ],
)

## 2. Validation And Test Leaderboards

`val_best` comes from `train.py`, while `test` comes from `test.py`.

In [ ]:
val_rows = sorted_stage_rows(active_rows, "val_best")
test_rows = sorted_stage_rows(active_rows, "test")

print("Validation-best rows")
display_rows(
    val_rows,
    columns=[
        "label",
        "metric_iou",
        "metric_f1",
        "metric_loss",
        "best_epoch",
        "img_size",
        "augmentation_profile",
        "foreground_crop_prob",
        "loss_name",
        "checkpoint_path",
    ],
)

print("Test rows")
display_rows(
    test_rows,
    columns=[
        "label",
        "metric_iou",
        "metric_f1",
        "metric_loss",
        "img_size",
        "augmentation_profile",
        "foreground_crop_prob",
        "loss_name",
        "checkpoint_path",
    ],
)

## 3. Metric Plots

These charts make it easy to scan whether a new `round1` setting is actually moving `IoU` / `F1`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

plot_stage_metric(axes[0, 0], active_rows, "val_best", "metric_iou", "Validation IoU")
plot_stage_metric(axes[0, 1], active_rows, "val_best", "metric_f1", "Validation F1")
plot_stage_metric(axes[1, 0], active_rows, "test", "metric_iou", "Test IoU")
plot_stage_metric(axes[1, 1], active_rows, "test", "metric_f1", "Test F1")

plt.tight_layout()
plt.show()

## 4. Delta Vs Older Baselines

If non-round1 `U-Net` rows already exist in the CSV, this section compares the current round against the best older baseline.

In [ ]:
baseline_test = best_baseline(latest_all_rows, stage="test", split="test")
baseline_val = best_baseline(latest_all_rows, stage="val_best", split="val")

if baseline_val:
    print(
        f"Best older validation baseline: {baseline_val['label']} | "
        f"IoU={baseline_val['metric_iou']:.4f} | F1={baseline_val['metric_f1']:.4f}"
    )
else:
    print("No older validation baseline row found yet.")

if baseline_test:
    print(
        f"Best older test baseline: {baseline_test['label']} | "
        f"IoU={baseline_test['metric_iou']:.4f} | F1={baseline_test['metric_f1']:.4f}"
    )
else:
    print("No older test baseline row found yet.")

test_delta_rows = []
if baseline_test:
    for row in test_rows:
        test_delta_rows.append(
            {
                "label": row["label"],
                "metric_iou": row["metric_iou"],
                "metric_f1": row["metric_f1"],
                "baseline_iou": baseline_test["metric_iou"],
                "baseline_f1": baseline_test["metric_f1"],
                "iou_delta_vs_baseline": (row["metric_iou"] or 0.0) - (baseline_test["metric_iou"] or 0.0),
                "f1_delta_vs_baseline": (row["metric_f1"] or 0.0) - (baseline_test["metric_f1"] or 0.0),
            }
        )

display_rows(
    test_delta_rows,
    columns=[
        "label",
        "metric_iou",
        "baseline_iou",
        "iou_delta_vs_baseline",
        "metric_f1",
        "baseline_f1",
        "f1_delta_vs_baseline",
    ],
)

## 5. Configuration View

This compact table is useful when you want to line up the result with the knobs we changed in round 1.

In [ ]:
display_rows(
    sorted(active_rows, key=lambda row: (row.get("stage", ""), -(row.get("metric_iou") or -1e9))),
    columns=[
        "label",
        "stage",
        "split",
        "img_size",
        "augmentation_profile",
        "foreground_crop_prob",
        "foreground_crop_min_scale",
        "foreground_crop_max_scale",
        "loss_name",
        "pos_weight",
        "tversky_alpha",
        "tversky_beta",
        "tversky_gamma",
        "metric_iou",
        "metric_f1",
    ],
)